In [14]:
import os
from dotenv import load_dotenv
# This function helps us to load all the environment variable from .env file to current notebook
load_dotenv()

True

In [15]:
if os.environ['OPENAI_API_KEY']:
    print("API key is set. ")

API key is set. 


In [16]:
from langchain_openai import ChatOpenAI

In [17]:
#Initiating LLM
llm = ChatOpenAI(model="gpt-5-nano", temperature =0)

In [18]:
#response = llm.invoke("What is AI, tell me in 1 line")
#response.content

## **RAG IMPLEMENTATION WITH YOUR OWN TEXT DATA**

#### **STEP 1 : PREPARING DOCUMENT FOR YOUR TEXT**

In [19]:
from langchain_core.documents import Document

In [20]:
# Your text data 
my_text = """Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]

High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.

The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers have used techniques including state space search and mathematical optimization, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind and Meta, aim to create artificial general intelligence (AGI) – AI that can complete virtually any cognitive task at least as well as a human.[3]"""

#This is the function that langchain uses internally to create document for our text
docs = [Document(page_content=my_text, metadata= {"source": "ABC", "documentID":"Doc1"})]

docs

[Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]\n\nHigh-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.\n\nThe traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language process

#### **STEP 2 : SPLITTING THE DOCUMENT INTO CHUNKS**

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50)  
# chunk_overlap is for the scenario where an information is split accross two different chunks, 
# hence chunk_overlap allows the chunk to pull 50 characters from the previous chunk so that the flow is not broken


chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.'),
 Document(metadata={'source': 'ABC', 'documentID': '

#### **STEP 3 : CREATING EMBEDDINGS FOR THE CHUNKS**

In [22]:
from langchain_openai import OpenAIEmbeddings

In [23]:
embedding_model = OpenAIEmbeddings()
#model ="text-embedding-5-small"

In [24]:
# To convert any input/prompt to embedding
#embedding_model.embed_query("What is AI?")

#### **STEP 4 : CREATE AND STORE EMBEDDINGS IN VECTORE STORE**

In [25]:
# Whenever we will be creating any vectore store, it will create the embedding for each chunk and store it in vector store automatically

In [40]:
#from langchain_chroma import Chroma

In [41]:
from langchain_community.vectorstores import Chroma

In [46]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

# User provided Chunks + Embedding Model to LangChain 
# LangChain converted them to embeddings and then these embeddings are stored in Chroma DB
# User --> Chunks + Embedding Model --> LangChain --> Embeddings --> Chroma DB

# Behind the scenes LangChain has done this 
# vectors = []

#for doc in chunks:
#    vector = embedding_model.embed_documents([doc.page_content])
#    vectors.append(vector)
   
# Note : ChromaDB has vector embedding along with the mapped chunk (content & metadata)  

#          CHROMA DB
# VECTOR            MAPPED CHUNK 
# [1,2,3,3,....] --> Document(metadata ={}, page_content="...")
# [1,9,4,3,....] --> Document(metadata ={}, page_content="...")
# [1,4,3,5,....] --> Document(metadata ={}, page_content="...")

#### **STEP 5 : SEMANTIC SEARCH**

In [53]:
vectorstore.similarity_search("What is AI?",k=3)  #For top 3 mapped chunks for the vectors

[Document(metadata={'documentID': 'Doc1', 'source': 'ABC'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machi

#### **TALK TO LLM**

In [54]:
context = vectorstore.similarity_search("What is AI?",k=3)

In [57]:
response = llm.invoke(f"What is AI? You can answer using the following context: {context}")
response.content

'AI is the capability of computational systems to perform tasks typically associated with human intelligence—such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops methods and software enabling machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.'